# **Pipeline Multiagente de Ciencia de Datos**

## Resolución de problemáticas usando modelos de IA
 
**Propósito:** Material educativo para introducir el concepto de sistemas multiagente aplicados a ciencia de datos.

### Estructura del notebook

Este notebook sigue un recorrido progresivo desde los fundamentos de la IA generativa hasta la implementación de un sistema multiagente funcional:

1. **Introducción a la IA generativa** — conceptos, historia y modelos de lenguaje.
2. **Agentes de IA** — qué son, cómo se diferencian de un LLM y sus componentes.
3. **Tecnologías para implementar agentes** — Ollama, LangChain y ejecución local.
4. **Diseño de un sistema multiagente** — arquitectura, orquestador y validación.
5. **Dataset y pipeline** — explicación del caso de estudio y las cuatro etapas.
6. **Discusión y mejoras futuras** — limitaciones y caminos de evolución.
7. **Conclusiones** — logros, cumplimiento del objetivo y valor educativo.
8. **Referencias bibliográficas**.

## **1. Introducción a la IA Generativa**

### 1.1 Concepto de IA generativa

La **inteligencia artificial generativa** es una rama de la IA capaz de crear contenidos originales (como texto, imágenes, vídeo, audio o código de software) en respuesta a las instrucciones o peticiones de un usuario.. A diferencia de los modelos *discriminativos* (que clasifican o predicen sobre datos existentes), los modelos generativos **crean** salidas nuevas condicionadas a una entrada o instrucción.

Los modelos de IA generativa han evolucionado a lo largo de la última docena de años aproximadamente (IBM, 2024). Las arquitecturas de modelos que han marcado un hito durante ese periodo incluyen:
- **Autocodificadores variacionales (VAE)**, que impulsaron grandes avances en el reconocimiento de imágenes, el procesamiento del lenguaje natural y la detección de anomalías.
- **Las redes generativas adversativas (GAN)** y **los modelos de difusión**, que mejoraron la precisión de las aplicaciones anteriores y permitieron algunas de las primeras soluciones de IA para la generación de imágenes fotorrealistas.
- **Transformers**, la arquitectura de modelos de deep learning que está detrás de los modelos fundacionales y las soluciones de IA generativa más importantes de la actualidad, ya que procesan secuencias de tokens mediante mecanismos de atención, permitiendo capturar dependencias de largo alcance en el lenguaje natural.

### 1.2 Línea del tiempo: origen, evolución y avances recientes

| Año | Hito | Institución |
|-----|------|-------------|
| 2017 | Publicación de *Attention Is All You Need* (arquitectura Transformer) | Google (y Univ. de Toronto) |
| 2018 | GPT-1 y BERT — modelos de lenguaje pre-entrenados a gran escala basados en Transformer | OpenAI / Google |
| 2019 | GPT-2 (1.5B parámetros) — demuestra generación de texto coherente | OpenAI |
| 2022 | ChatGPT (noviembre) populariza la IA conversacional | OpenAI |
| 2023 | GPT-4, LLaMA, Claude, Mistral 7B — modelos abiertos y comerciales conviven | OpenAI, Meta, Anthropic, Mistral AI |
| 2024 | LLaMA 3, Claude 3, Qwen2.5 — modelos más eficientes y especializados; o1 inaugura los modelos entrenados para razonar antes de responder | Meta, Anthropic, Alibaba, OpenAI |
| 2025 | DeepSeek-R1 y Qwen3 — razonamiento extendido también en modelos de pesos abiertos | DeepSeek, Alibaba |
| 2026 | GPT-5.4, Gemini 3.1, Kimi K2.5 — modelos enfocado en agentes auntónomos y código más robusto | OpenAI, Google y Mooonshot AI |

> La tabla es ilustrativa, no exhaustiva: omite muchos modelos y versiones intermedias.

### 1.3 ¿Qué es un modelo de lenguaje grande (LLM)?

Un **Large Language Model (LLM)** es una red neuronal entrenada, en su fase base, para predecir el siguiente token de una secuencia de texto. Sus características principales son:

- **Escala**: miles de millones de parámetros (típicamente entre 1B y 400B o más).
- **Pre-entrenamiento**: se entrena sobre grandes corpus de texto (web, libros, código).
- **Ajuste (fine-tuning)**: se especializa con instrucciones, diálogo, preferencias humanas o dominios específicos.

### 1.4 Resolución de problemas mediante IA generativa

Los LLMs resuelven problemas mediante **inferencia condicional**: dada una instrucción (*prompt*), generan una secuencia de tokens que constituye la respuesta. Esto permite abordar tareas diversas (resumen, clasificación, generación de código, análisis de datos) con un único modelo, siempre que la tarea pueda expresarse en lenguaje natural o en un formato estructurado (como JSON).

## **2. Agentes de inteligencia artificial**

### 2.1 ¿Qué es un agente y en qué se diferencia de un LLM?

Un **agente** es un sistema que percibe su entorno, razona sobre él y actúa para lograr un objetivo (Wooldridge & Jennings, 1995; Russell & Norvig, 2021). Algunos agentes además *aprenden* de los resultados, pero eso no es un requisito de la definición. Un LLM es sólo el componente de razonamiento; un agente lo integra con herramientas, memoria y un ciclo de decisión.

| Aspecto | LLM | Agente |
|---------|-----|--------|
| Entrada | Prompt aislado | Contexto, herramientas, estado |
| Salida | Texto o JSON | Acciones sobre el entorno |
| Memoria | Sin estado entre llamadas | Mantiene historial y decisiones |
| Herramientas | No tiene | Puede invocar código, APIs, bases de datos |
| Autonomía | Responde cuando se le pregunta | Decide cuándo y cómo actuar |

### 2.2 Tipos y arquitecturas de agentes

Clasificación clásica de Russell & Norvig (2021):

- **Agente reactivo simple**: responde directamente a percepciones actuales sin necesidad de una planificación compleja o considerar historial.
- **Agente reactivo basado en modelos**: mantiene un estado interno del mundo.
- **Agente basado en objetivos**: planifica acciones para alcanzar un fin.
- **Agente basado en utilidad**: elige las acciones que maximizan una medida de utilidad (comparativo).
- **Agente con aprendizaje**: mejora su comportamiento con la experiencia.

En el contexto de LLMs, los agentes modernos suelen combinar razonamiento en cadena (*chain-of-thought*; Wei et al., 2022), uso de herramientas (*tool use*) y memoria a corto y largo plazo.

> **Agentes frente a flujos de trabajo (*workflows*).** Anthropic (2024) distingue entre *workflows* —donde el código orquesta las llamadas al LLM por rutas predefinidas— y *agentes*, donde el propio LLM dirige dinámicamente su proceso y el uso de herramientas. El sistema de este notebook se ubica más cerca del primer caso: los scripts de cada etapa son deterministas y el LLM propone planes e interpreta resultados, mientras que sólo el supervisor decide dinámicamente el siguiente paso. Aun así es útil como introducción a los sistemas multiagente, porque comparte sus ideas centrales: roles especializados, contratos de comunicación y supervisión.

### 2.3 Elementos de un agente

1. **Modelo**: el LLM que realiza el razonamiento (ej. Qwen3.5, GPT-4).
2. **Prompt**: instrucciones que definen el rol, las reglas y el formato de salida.
3. **Contexto**: información relevante disponible en el momento de la decisión.
4. **Herramientas**: funciones que el agente puede invocar (código Python, APIs, bases de datos).
5. **Memoria**: historial de interacciones y decisiones previas.
6. **Estado**: representación actual del entorno y del progreso del agente.

### 2.4 Conceptos complementarios

- **Tokens**: unidades mínimas de texto (palabras, subpalabras o caracteres). En inglés, un token equivale en promedio a unos 4 caracteres, es decir, a ¾ de palabra. En español, al ser menos frecuente en los datos de entrenamiento, se necesitan algo más de tokens por palabra (aprox. entre 3 y 4 caracteres por token, según el tokenizador). Ejemplo: un texto de 1 000 palabras en español puede consumir entre 1 500 y 2 000 tokens.
- **Ventana de contexto**: cantidad máxima de tokens que el modelo puede procesar en una sola llamada (típicamente 4K, 8K, 32K, 128K o más; Qwen3.5 declara 262K).
- **Temperatura**: parámetro que controla la aleatoriedad de la salida. Valores bajos (0.0-0.3) producen respuestas más deterministas; valores altos (0.7-1.0) producen respuestas más variadas. Incluso con temperatura baja, la salida no está garantizada como idéntica entre corridas.
- **Salida estructurada**: forzar al modelo a responder en un formato específico (JSON, XML) para facilitar el procesamiento automático.

### 2.5 Ejemplos de modelos

| Modelo | Desarrollador | Año | Parámetros | Acceso / licencia |
|--------|---------------|-----|------------|-------------------|
| GPT-4 | OpenAI | 2023 | No divulgado (se rumoran ~1.8T) | Comercial, cerrado |
| Claude 3 | Anthropic | 2024 | No divulgado | Comercial, cerrado |
| LLaMA 3 | Meta | 2024 | 8B, 70B (405B en Llama 3.1) | Pesos abiertos, licencia propia de Meta (con restricciones de uso) |
| Mistral 7B | Mistral AI | 2023 | 7B | Pesos abiertos (Apache 2.0) |
| Qwen2.5 | Alibaba | 2024 | 0.5B – 72B | Pesos abiertos (la licencia varía según el tamaño) |
| Qwen3 | Alibaba | 2025 | 0.6B – 235B | Pesos abiertos (Apache 2.0) |
| Qwen3.5 | Alibaba | 2026 | 0.8B – 397B (los modelos grandes usan mezcla de expertos) | Pesos abiertos (Apache 2.0) |

### 2.6 Modelos comerciales, de pesos abiertos y plataformas gratuitas

- **Comerciales cerrados** (GPT-4, Claude): acceso vía API, costo por token, normalmente el mayor rendimiento.
- **Pesos abiertos** (LLaMA, Qwen, Mistral): los pesos del modelo pueden descargarse y ejecutarse localmente, sin costo por uso. **Pesos abiertos no equivale a código abierto**: la licencia determina qué usos están permitidos, por lo que debe revisarse en cada caso.
- **Plataformas de acceso y cómputo gratuito**: Hugging Face (repositorio de modelos y conjuntos de datos) y Google Colab (entorno de cómputo con GPU limitada) permiten experimentar sin costo, con límites de uso.

En este proyecto se usan modelos de **pesos abiertos** ejecutados localmente con Ollama, lo que favorece la privacidad, el control de versiones y el costo cero por inferencia.

## **3. Tecnologías para implementar agentes**

### 3.1 Inferencia de modelos de lenguaje

La **inferencia** es el proceso de usar un modelo entrenado para generar salidas a partir de nuevas entradas. Requiere:

- **Hardware adecuado**: memoria suficiente (VRAM de la GPU y/o RAM) para cargar el modelo. Regla práctica: un modelo cuantizado a 4 bits ocupa ~0.5-0.6 GB por cada mil millones de parámetros, más memoria para el contexto. Ejemplos en Ollama: `qwen2.5:7b` ≈ 4.7 GB y `qwen3.5:9b` ≈ 6.6 GB.
- **Framework de inferencia**: software que gestiona la carga del modelo y la generación de tokens.
- **Interfaz de comunicación**: API local o remota para enviar prompts y recibir respuestas.

### 3.2 Ollama: ejecución local de LLM

**Ollama** es una herramienta de código abierto (2023) que permite ejecutar LLMs localmente en macOS, Linux y Windows. Características:

- Descarga y gestiona modelos desde un registro centralizado.
- Distribuye los modelos ya cuantizados (reducción de la precisión numérica de los pesos) para que ocupen menos memoria.
- Permite configurar parámetros como `num_ctx` (ventana de contexto) y `num_predict` (límite de tokens de salida).

### 3.3 LangChain: integración de LLM con Python y herramientas

**LangChain** es un framework de Python (creado por Harrison Chase en 2022) que facilita la construcción de aplicaciones con LLMs. Proporciona:

- **Integradores**: conectores para Ollama, OpenAI, Anthropic, Hugging Face, etc.
- **Cadenas (chains)**: composición de pasos de procesamiento.
- **Agentes**: sistemas que pueden usar herramientas de forma autónoma.
- **Memoria**: gestión de historial de conversaciones.

En este proyecto se usa `langchain-ollama` para crear los clientes LLM y `langchain-core` para manejar los mensajes del sistema y del usuario.

### 3.4 Ejecución local frente a nube y servidores

| Aspecto | Local (Ollama) | Nube (API comercial) |
|---------|----------------|----------------------|
| Costo | Sin costo por uso (tras la inversión en hardware) | Pago por token |
| Privacidad | Alta (los datos no salen del equipo) | Los datos viajan a servidores del proveedor |
| Latencia | Sin latencia de red, pero depende del hardware | Depende de la conexión y la carga del proveedor |
| Escalabilidad | Limitada por el hardware | Elástica |
| Modelos disponibles | Los de pesos abiertos | Los que ofrece cada proveedor (propietarios y, en algunos casos, abiertos) |

### 3.5 Selección de modelos según tareas y recursos

La elección del modelo depende de:

- **Tarea**: tareas simples (clasificación, extracción) → modelos pequeños; tareas complejas (razonamiento, planificación) → modelos más grandes.
- **Recursos**: VRAM disponible, tiempo de inferencia aceptable.
- **Costo**: si hay presupuesto, modelos comerciales; si no, modelos abiertos locales.

En este proyecto se usa un esquema de **dos modelos**:
- `qwen2.5:7b` para etapas de análisis y ejecución (tareas moderadas).
- `qwen3.5:9b` para etapas de razonamiento profundo (EDA, planificación, supervisión).

## **4. Diseño de un sistema multiagente**

### 4.1 Concepto y características de los sistemas multiagente

Un **sistema multiagente (MAS)** es un conjunto de agentes que interactúan para resolver problemas que un solo agente resolvería con dificultad. Características:

- **Autonomía**: cada agente decide dentro de su dominio.
- **Especialización**: cada agente tiene un rol específico.
- **Comunicación**: los agentes intercambian información mediante mensajes estructurados.
- **Coordinación**: un mecanismo (orquestador, supervisor) gestiona el flujo de trabajo.

### 4.2 Diferencias entre un agente individual y un sistema multiagente

| Aspecto | Agente individual | Sistema multiagente |
|---------|-------------------|---------------------|
| Complejidad | Tarea única, contexto único | Tarea compleja dividida en subtareas con contextos más pequeños |
| Robustez | Un fallo detiene todo | Permite reintentar o aislar fallos por etapa (aunque un error puede propagarse a las etapas siguientes) |
| Escalabilidad | Limitada | Modular |
| Trazabilidad | Depende del registro implementado | Facilitada si cada agente registra sus decisiones |
| Costo | Una llamada o pocas | Más llamadas al LLM: mayor tiempo y consumo de recursos |

### 4.3 Aplicación al flujo de trabajo de ciencia de datos

Un pipeline de ciencia de datos tiene etapas bien definidas: inspección, limpieza, análisis, modelado y evaluación. Cada etapa puede delegarse a un componente especializado:

- **Agente 1 (Inspección)**: analiza el dataset y propone un plan.
- **Agente 2 (Estadística)**: ejecuta la limpieza propuesta y calcula estadísticas.
- **Agente 3 (EDA y modelos)**: diseña la estrategia de modelado.
- **Agente 4 (Entrenamiento)**: entrena y evalúa los modelos.
- **Supervisor**: coordina el flujo, valida contratos y decide cuándo avanzar o corregir.

## **5. Explicación del dataset elegido para ejemplificar este pipeline**

### 5.1 Propuesta de las cuatro etapas del pipeline

| Etapa | Actividades del agente | Detalles del prompt | Archivos de entrada | Archivos de salida | Mejoras / Consideraciones |
|-------|------------------------|---------------------|---------------------|--------------------|--------------------------|
| **1. Inspección** | Analiza el dataset, define el problema, propone limpieza y ruido | Se entrega el perfil del dataset, la procedencia y las candidatas a objetivo. El agente decide la variable objetivo y la tarea ML. | `data/data.csv` | `data/metadata.json`, `data/reports/reporte_inicial.md` | Soporte para clasificación (actualmente sólo regresión). |
| **2. Estadística** | Ejecuta la limpieza propuesta, aplica ruido, calcula estadísticas e interpreta | Recibe las propuestas de Etapa 1 y los hechos calculados por Python. Interpreta distribuciones, outliers y correlaciones. | `data/metadata.json`, `data/data.csv` | `data/dataset_experimental.csv`, `data/registro_limpieza.json`, `data/registro_ruido.json`, `data/estadisticas.json`, `data/reports/reporte_estadistico.md` | Reducir el tamaño del prompt para evitar truncamiento del JSON. |
| **3. EDA y modelos** | EDA profundo, línea base y diseño de estrategia experimental | Recibe hechos de cardinalidad, distribución, fugas y línea base. Decide transformación, encoding, escalado, modelos y métricas. | `data/metadata.json`, `data/estadisticas.json`, `data/dataset_experimental.csv` | `data/propuesta_modelos.json`, `data/plan_experimental.json`, `data/reports/reporte_eda.md` | Añadir regla de dimensionalidad para evitar sobreajuste por one-hot. |
| **4. Entrenamiento** | Entrena los modelos del plan, calcula métricas, compara con línea base e interpreta | Recibe el plan experimental y ejecuta el entrenamiento. Interpreta resultados en contexto de negocio. | `data/plan_experimental.json`, `data/propuesta_modelos.json`, `data/dataset_experimental.csv` | `data/metricas.json`, `data/comparacion_modelos.csv`, `data/reports/reporte_resultados.md`, `models/*.joblib` | Añadir explicabilidad (SHAP, LIME) y cross-validation. |

### 5.2 Descripción del dataset, origen y licencia

El dataset proviene del repositorio **[ticketmaster-price-ml](https://github.com/ethanjaredlee/ticketmaster-price-ml)** de **ethanjaredlee**, publicado bajo **licencia MIT** (copyright 2018). El repositorio contiene varios archivos CSV y scripts de recolección; este proyecto usa `data.csv`.

**Características del archivo csv:**
- **Filas:** 1198
- **Columnas:** 9
- **Tarea original:** predicción del precio mínimo de eventos musicales.

**Columnas:**

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `city` | Categórica (203 únicos) | Ciudad del evento |
| `artist` | Categórica (87 únicos) | Artista |
| `venue` | Categórica (335 únicos) | Lugar del evento |
| `weekend` | Binaria (0/1) | Si es fin de semana |
| `pop` | Numérica (con valores `???`) | Popularidad o tamaño de audiencia estimado* |
| `month` | Ordinal (1-12) | Mes del evento |
| `score` | Numérica (65-100) | Puntuación asociada al evento* |
| `genre` | Categórica (18 únicos) | Género musical |
| `minprice` | Numérica (1-2999) | Precio mínimo del boleto |

**Consideraciones de calidad:**
- 17 valores nulos en `venue`.
- 28 filas duplicadas.
- Valores `???` en `pop` (columna casi numérica con corrupción).
- Alta cardinalidad en `city`, `venue` y `artist`: con 1198 filas y 335 lugares distintos, cada lugar aparece en promedio unas 3.6 veces, lo que genera riesgo de sobreajuste con one-hot.

## **6. Implementación del orquestador**

El orquestador es un **notebook de Jupyter** que coordina las cuatro etapas. Su funcionamiento:

1. **Carga la configuración**: modelos, reintentos máximos, archivo de log.
2. **Define los contratos**: qué archivos y campos JSON debe generar cada etapa.
3. **Ejecuta cada etapa** como subproceso independiente (los scripts `etapa1_...py` a `etapa4_...py` y el módulo `comun.py` se encuentran fuera de este notebook).
4. **Valida el contrato** de salida (archivos presentes, campos JSON, estado `EXITO`).
5. **Consulta al supervisor LLM** para decidir la siguiente acción, entregándole el resultado de la validación.
6. **Registra todas las decisiones** en `orquestador_log.json`.

**Alcance de la validación:** el código comprueba que la acción elegida por el supervisor pertenezca al conjunto de acciones legales. El resultado de la validación del contrato se entrega al supervisor como evidencia para su decisión, pero no obliga a una acción concreta.

El supervisor puede decidir:
- `RUN_NEXT_STAGE`: avanzar a la siguiente etapa.
- `RETRY_STAGE`: reintentar la etapa actual.
- `REQUEST_CORRECTION`: pedir corrección al agente.
- `STOP_SUCCESS`: pipeline completado.
- `STOP_FAILURE`: detener por error crítico.
- `REQUEST_HUMAN_REVIEW`: pedir intervención humana.

In [17]:
# Imports y configuración
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

# Configuración centralizada (dos modelos distintos para diferentes etapas del pipeline)
MODEL_FAST = "qwen2.5:7b" # Etapas 1 y 2 (tareas ligeras)
MODEL_SMART = "qwen3.5:9b" # Orquestador, Etapas 3 y 4 (razonamiento)

MAX_RETRIES_PER_STAGE = 5 
LOG_FILE = Path("data/orquestador_log.json")

# Acciones que el supervisor puede decidir
ACCIONES_VALIDAS = {
    "RUN_NEXT_STAGE", # Avanzar a la siguiente etapa
    "REQUEST_CORRECTION", # Pedir corrección al agente actual
    "RETRY_STAGE", # Reintentar la etapa actual
    "STOP_SUCCESS", # Pipeline completado con éxito
    "STOP_FAILURE", # Detener por error crítico
    "REQUEST_HUMAN_REVIEW", # Pedir intervención humana (usar archivo de revisión)
}

print("Configuración cargada con éxito:")
print(f"- Modelo rápido (etapas 1-2): {MODEL_FAST}")
print(f"- Modelo inteligente (etapas 3-4): {MODEL_SMART}")

Configuración cargada con éxito:
- Modelo rápido (etapas 1-2): qwen2.5:7b
- Modelo inteligente (etapas 3-4): qwen3.5:9b


### 6.1. Definición de Etapas y Contratos

Cada etapa tiene:
- **Script Python** que ejecuta la tarea.
- **Modelo LLM** asignado según complejidad.
- **Salidas requeridas** (archivos que debe generar).
- **Campos JSON requeridos** (contrato de comunicación).

El orquestador valida que cada etapa cumpla su contrato antes de avanzar.

In [18]:
# Definición del pipeline y sus contratos mínimos de salida.Y
STAGES = [
    {
        "id": "etapa1_inspeccion",
        "script": "etapa1_inspeccion.py",
        "modelo": MODEL_FAST,
        "salidas_requeridas": [
            "data/metadata.json",
            "data/reports/reporte_inicial.md",
        ],
        "campos_json_requeridos": [
            "etapa", "estado", "timestamp",
            "descripcion_semantica",
            "definicion_problema.variable_objetivo",
            "definicion_problema.tarea_ml",
            "propuestas_limpieza",
            "propuesta_ruido",
        ],
    },
    {
        "id": "etapa2_estadistica",
        "script": "etapa2_estadistica.py",
        "modelo": MODEL_FAST,
        "salidas_requeridas": [
            "data/estadisticas.json",
            "data/dataset_experimental.csv",
            "data/registro_limpieza.json",
            "data/registro_ruido.json",
            "data/reports/reporte_estadistico.md",
        ],
        "campos_json_requeridos": [
            "etapa", "estado", "timestamp", "variable_objetivo",
            "limpieza_ejecutada", "ruido_artificial",
            "estadisticas_descriptivas", "interpretacion_llm",
        ],
    },
    {
        "id": "etapa3_eda_modelos",
        "script": "etapa3_eda_modelos.py",
        "modelo": MODEL_SMART,
        "salidas_requeridas": [
            "data/propuesta_modelos.json",
            "data/plan_experimental.json",
            "data/reports/reporte_eda.md",
        ],
        "campos_json_requeridos": [
            "etapa", "estado", "timestamp", "variable_objetivo",
            "plan_normalizado.modelos_propuestos",
            "plan_normalizado.metricas_evaluacion",
            "plan_normalizado.estrategia_validacion",
        ],
    },
    {
        "id": "etapa4_entrenamiento",
        "script": "etapa4_entrenamiento.py",
        "modelo": MODEL_SMART,
        "salidas_requeridas": [
            "data/metricas.json",
            "data/comparacion_modelos.csv",
            "data/reports/reporte_resultados.md",
        ],
        "campos_json_requeridos": [
            "etapa", "estado", "timestamp", "variable_objetivo",
            "resultados_por_modelo", "modelo_ganador",
            "interpretacion_llm.limitaciones",
        ],
    },
]

print(f"Pipeline definido con {len(STAGES)} etapas")
for i, stage in enumerate(STAGES, 1):
    print(f" -> Etapa {i}: {stage['id']} ({stage['modelo']})")

Pipeline definido con 4 etapas
 -> Etapa 1: etapa1_inspeccion (qwen2.5:7b)
 -> Etapa 2: etapa2_estadistica (qwen2.5:7b)
 -> Etapa 3: etapa3_eda_modelos (qwen3.5:9b)
 -> Etapa 4: etapa4_entrenamiento (qwen3.5:9b)


### 6.2 El Supervisor LLM

El supervisor es un agente que:
- **Analiza** el estado del pipeline (archivos generados, errores, métricas).
- **Decide** la siguiente acción (avanzar, reintentar, corregir, detener).
- **Justifica** su decisión con evidencia concreta.

### ¿Por qué un LLM como supervisor?

Porque puede interpretar contexto ambiguo. Por ejemplo:
- Si una etapa falla pero genera archivos parciales, ¿debe reintentar o pedir corrección?
- Si el JSON tiene todos los campos pero con valores vacíos, ¿es válido?
- Si hay un error de codificación, ¿es crítico o recuperable?

Un script tradicional no puede razonar sobre estos casos. Un LLM sí.

In [19]:
# Utilidades
from comun import asegurar_directorios, limpiar_json

def cargar_log():
    """Carga el historial de decisiones del orquestador."""
    if LOG_FILE.exists():
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"decisiones": []}

def guardar_log(log):
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)

def registrar_decision(decision: dict, resultado: dict):
    """Añade una entrada al log con la decisión del supervisor y su resultado."""
    log = cargar_log()
    log["decisiones"].append({
        "timestamp": datetime.now().isoformat(),
        "decision_supervisor": decision,
        "resultado_ejecucion": resultado,
    })
    guardar_log(log)

def obtener_campo_anidado(datos: dict, ruta: str) -> tuple:
    """Obtiene un campo de un dict usando notación de puntos.
    Ej: 'definicion_problema.variable_objetivo'.
    Devuelve (valor, existe)."""
    claves = ruta.split(".")
    actual = datos
    for clave in claves:
        if not isinstance(actual, dict) or clave not in actual:
            return None, False
        actual = actual[clave]
    return actual, True

def validar_contrato(etapa: dict) -> tuple[bool, str]:
    """Verifica que existan las salidas requeridas, que el JSON principal
    contenga los campos mínimos y que el estado sea EXITO."""
    # 1. Verificar archivos
    for archivo in etapa["salidas_requeridas"]:
        if not Path(archivo).exists():
            return False, f"Falta archivo requerido: {archivo}"

    # 2. Verificar el JSON principal de la etapa
    json_principal = next(
        (a for a in etapa["salidas_requeridas"] if a.endswith(".json")),
        None,
    )
    if json_principal:
        try:
            with open(json_principal, "r", encoding="utf-8") as f:
                datos = json.load(f)

            # Verificar estado 
            estado = datos.get("estado")
            if estado is not None and estado != "EXITO":
                return False, f"La etapa reportó estado '{estado}' (no 'EXITO')"

            # Verificar campos requeridos
            campos_faltantes = []
            for ruta in etapa["campos_json_requeridos"]:
                _, existe = obtener_campo_anidado(datos, ruta)
                if not existe:
                    campos_faltantes.append(ruta)

            if campos_faltantes:
                return False, f"JSON incompleto. Faltan campos: {campos_faltantes}"
        except json.JSONDecodeError as e:
            return False, f"JSON inválido en {json_principal}: {e}"

    return True, "OK"

print("Funciones utilitarias definidas")

Funciones utilitarias definidas


### 6.3 Ejecución de Etapas

Cada etapa se ejecuta como un **subproceso independiente**. Esto tiene ventajas:
- **Aislamiento**: Si una etapa falla, no contamina las demás.
- **Reproducibilidad**: Cada etapa puede ejecutarse por separado.
- **Debugging**: Se puede inspeccionar el output de cada etapa.

### ¿Por qué no ejecutar todo en el mismo proceso?

Porque los agentes deben ser **independientes**. Si comparten memoria, no son agentes, son funciones. La comunicación por archivos fuerza la trazabilidad y la reproducibilidad.

In [20]:
# Ejecución de etapas
def ejecutar_etapa(etapa: dict) -> dict:
    """Ejecuta el script Python de la etapa como subproceso."""
    script = etapa["script"]
    modelo = etapa["modelo"]
    print(f"\n{'=' * 80}")
    print(f"[ORQUESTADOR] Ejecutando {script} con modelo {modelo}")
    print(f"{'=' * 80}")

    try:
        resultado = subprocess.run(
            [sys.executable, script, "--modelo", modelo],
            capture_output=True,
            text=True,
            timeout=1800,  # 30 minutos máximo por etapa
            check=False,
            encoding="utf-8",
            errors="replace",
        )
        
        # Imprimir output en tiempo real para debugging
        if resultado.stdout:
            print(resultado.stdout[-1000:])
        if resultado.stderr:
            print(f"[STDERR] {resultado.stderr[-500:]}")
        
        return {
            "returncode": resultado.returncode,
            "stdout": resultado.stdout[-2000:],
            "stderr": resultado.stderr[-2000:],
        }
    except subprocess.TimeoutExpired:
        return {
            "returncode": -1,
            "stdout": "",
            "stderr": "TIMEOUT: la etapa excedió los 30 minutos.",
        }
    except Exception as e:
        return {
            "returncode": -1,
            "stdout": "",
            "stderr": f"Error al lanzar subproceso: {e}",
        }

print("Función de ejecución definida")

Función de ejecución definida


### 6.4 Consulta al Supervisor LLM

El supervisor recibe un **contexto estructurado** con:
- Etapa actual y su índice.
- Resultado de ejecución (código de retorno, errores).
- Validación del contrato (archivos presentes, campos JSON).
- Intentos realizados.

Y devuelve una **decisión estructurada** en JSON con:
- Acción a tomar.
- Agente afectado.
- Justificación.
- Evidencia en que se basa.
- Criterios de aceptación.

In [21]:
def crear_supervisor():
    """Instancia el LLM supervisor con contexto amplio y tokens de salida suficientes."""
    return ChatOllama(
        model=MODEL_SMART,
        temperature=0.1,
        format="json",
        num_ctx=16384, # evita truncamiento del prompt
        num_predict=4096, # evita truncamiento de la respuesta
    )

def consultar_supervisor(supervisor, contexto: dict) -> dict:
    """Pide al LLM que decida la siguiente acción."""
    system_prompt = (
        "Eres el supervisor de un pipeline de ciencia de datos multiagente. "
        "Tu trabajo es analizar el estado actual del pipeline y decidir la "
        "siguiente acción. Debes responder EXCLUSIVAMENTE con un JSON válido "
        "sin texto adicional, sin markdown, sin comentarios.\n\n"
        "Acciones permitidas:\n"
        "- RUN_NEXT_STAGE: avanzar a la siguiente etapa.\n"
        "- REQUEST_CORRECTION: pedir al agente actual que corrija su salida.\n"
        "- RETRY_STAGE: reintentar la etapa actual.\n"
        "- STOP_SUCCESS: pipeline completado con éxito.\n"
        "- STOP_FAILURE: detener por error crítico.\n"
        "- REQUEST_HUMAN_REVIEW: pedir intervención humana.\n\n"
        "Formato de respuesta JSON:\n"
        "{\n"
        '  "accion": "<una de las acciones permitidas>",\n'
        '  "agente_afectado": "<id de la etapa o null>",\n'
        '  "justificacion": "<razón breve de la decisión>",\n'
        '  "evidencia": "<archivos o métricas en que te basas>",\n'
        '  "criterios_aceptacion": "<qué debe cumplirse para considerar éxito>"\n'
        "}\n"
    )

    user_prompt = (
        f"Estado actual del pipeline:\n{json.dumps(contexto, indent=2, ensure_ascii=False)}\n\n"
        "Decide la siguiente acción. Responde solo con el JSON."
    )

    try:
        respuesta = supervisor.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        texto = limpiar_json(respuesta.content)
        decision = json.loads(texto)

        # Validar que la acción sea legal
        if decision.get("accion") not in ACCIONES_VALIDAS:
            decision["accion"] = "REQUEST_HUMAN_REVIEW"
            decision["justificacion"] += (
                " [Corregido por validador: acción ilegal detectada]"
            )
        return decision
    # Excepeción para manejo de errores de manera manual
    except (json.JSONDecodeError, AttributeError, KeyError) as e:
        return {
            "accion": "REQUEST_HUMAN_REVIEW",
            "agente_afectado": contexto.get("etapa_actual"),
            "justificacion": f"El supervisor no pudo generar JSON válido: {e}",
            "evidencia": "Error de parsing",
            "criterios_aceptacion": "Revisión manual requerida",
        }

print("Funciones del supervisor definidas")

Funciones del supervisor definidas


### 6.5 Máquina de Estados Principal

El orquestador es una **máquina de estados** que:
1. Ejecuta la etapa actual.
2. Valida el contrato de salida.
3. Consulta al supervisor LLM.
4. Actúa según la decisión (avanzar, reintentar, corregir, detener).
5. Registra todo en un log.

In [22]:
def construir_contexto(indice_etapa: int, resultado_ejecucion: dict,
                       intentos: int, validacion: tuple[bool, str]) -> dict:
    """Construye el contexto que se envía al supervisor LLM."""
    etapa = STAGES[indice_etapa]
    archivos_existentes = [
        str(p) for p in Path("data").rglob("*") if p.is_file()
    ][:50]

    return {
        "etapa_actual": etapa["id"],
        "indice_etapa": indice_etapa,
        "total_etapas": len(STAGES),
        "script": etapa["script"],
        "modelo_asignado": etapa["modelo"],
        "intentos_realizados": intentos,
        "max_intentos": MAX_RETRIES_PER_STAGE,
        "resultado_ejecucion": {
            "returncode": resultado_ejecucion["returncode"],
            "stderr_ultimas_lineas": resultado_ejecucion["stderr"][-500:],
        },
        "validacion_contrato": {
            "valido": validacion[0],
            "detalle": validacion[1],
        },
        "archivos_en_data": archivos_existentes,
    }

def ejecutar_pipeline():
    """Bucle principal del orquestador."""
    asegurar_directorios()
    supervisor = crear_supervisor()
    indice_etapa = 0
    intentos = 0

    print("[ORQUESTADOR] Iniciando pipeline multiagente para resolución de problemas de ciencia de datos.")
    print(f"[ORQUESTADOR] Modelo rápido: {MODEL_FAST}")
    print(f"[ORQUESTADOR] Modelo inteligente: {MODEL_SMART}")
    print(f"[ORQUESTADOR] Total de etapas: {len(STAGES)}\n")

    while indice_etapa < len(STAGES):
        etapa = STAGES[indice_etapa]

        # 1. Ejecutar la etapa
        resultado = ejecutar_etapa(etapa)

        # 2. Validar contrato
        if resultado["returncode"] == 0:
            validacion = validar_contrato(etapa)
        else:
            validacion = (False, f"Script falló con código {resultado['returncode']}")

        # 3. Consultar al supervisor
        contexto = construir_contexto(indice_etapa, resultado, intentos, validacion)
        decision = consultar_supervisor(supervisor, contexto)

        # 4. Registrar y actuar según la decisión
        registrar_decision(decision, resultado)
        print(f"\n[ORQUESTADOR] Decisión del supervisor: {decision['accion']}")
        print(f"  Justificación: {decision['justificacion']}")

        accion = decision["accion"]

        if accion == "RUN_NEXT_STAGE":
            indice_etapa += 1
            intentos = 0
            print(f"  -> Avanzando a etapa {indice_etapa + 1}/{len(STAGES)}")

        elif accion == "RETRY_STAGE":
            intentos += 1
            if intentos > MAX_RETRIES_PER_STAGE:
                print("  -> Máximo de reintentos alcanzado. Deteniendo.")
                decision["accion"] = "STOP_FAILURE"
                registrar_decision(decision, resultado)
                break
            print(f"  -> Reintentando etapa (intento {intentos}/{MAX_RETRIES_PER_STAGE})")

        elif accion == "REQUEST_CORRECTION":
            intentos += 1
            if intentos > MAX_RETRIES_PER_STAGE:
                print("  -> Correcciones agotadas. Deteniendo.")
                decision["accion"] = "STOP_FAILURE"
                registrar_decision(decision, resultado)
                break
            print(f"  -> Solicitando corrección al agente (intento {intentos})")

        elif accion == "STOP_SUCCESS":
            print("\n[ORQUESTADOR] Pipeline completado con éxito.")
            break

        elif accion == "STOP_FAILURE":
            print("\n[ORQUESTADOR] Pipeline detenido por error crítico.")
            break

        elif accion == "REQUEST_HUMAN_REVIEW":
            print("\n[ORQUESTADOR] Se requiere intervención humana.")
            print(f"  Razón: {decision['justificacion']}")
            break

        else:
            print(f"\n[ORQUESTADOR] Acción inesperada: {accion}. Deteniendo.")
            break

    print("\n[ORQUESTADOR] Fin de la ejecución.")
    print(f"[ORQUESTADOR] Log completo en: {LOG_FILE}")

print("Máquina de estados definida")

Máquina de estados definida


### 6.6 Ejecución del Pipeline

Ahora ejecutamos el pipeline completo. El orquestador coordinará las 4 etapas, validará contratos y registrará todas las decisiones.

In [23]:
# Ejecutar el pipeline
ejecutar_pipeline()

[ORQUESTADOR] Iniciando pipeline multiagente para resolución de problemas de ciencia de datos.
[ORQUESTADOR] Modelo rápido: qwen2.5:7b
[ORQUESTADOR] Modelo inteligente: qwen3.5:9b
[ORQUESTADOR] Total de etapas: 4


[ORQUESTADOR] Ejecutando etapa1_inspeccion.py con modelo qwen2.5:7b

[Etapa 1] Iniciando análisis con modelo: qwen2.5:7b
[Etapa 1] Dataset cargado (latin-1): (1198, 9)
[Etapa 1] Analizando semántica y definiendo el problema...
  -> Tarea: regresion
  -> Objetivo: score
[Etapa 1] Generando propuesta de limpieza...
[Etapa 1] Generando propuesta de ruido...

[Etapa 1] Completada. Archivos generados:
  -> data\metadata.json
  -> data\reports\reporte_inicial.md


[ORQUESTADOR] Decisión del supervisor: RUN_NEXT_STAGE
  Justificación: La etapa de inspección completó exitosamente con todos los archivos esperados generados y validación del contrato aprobada
  -> Avanzando a etapa 2/4

[ORQUESTADOR] Ejecutando etapa2_estadistica.py con modelo qwen2.5:7b
2.5:7b
[Etapa 2] Variable objet

### 6.7 Análisis de Resultados

Una vez completado el pipeline, podemos analizar:
- **Decisiones del supervisor**: ¿Cuándo pidió correcciones? ¿Cuántos reintentos necesitó?
- **Artefactos generados**: ¿Se generaron todos los archivos esperados?
- **Métricas de modelos**: ¿Qué modelo ganó? ¿Por qué?
- **Limitaciones**: ¿Qué problemas detectó el LLM?

#### Visualización del log de decisiones

In [24]:
# Mostrar el log de decisiones del orquestador
if LOG_FILE.exists():
    log = cargar_log()
    print(f"Total de decisiones registradas: {len(log['decisiones'])}\n")
    
    for i, decision in enumerate(log["decisiones"], 1):
        print(f"--- Decisión {i} ---")
        print(f"Timestamp: {decision['timestamp']}")
        print(f"Acción: {decision['decision_supervisor']['accion']}")
        print(f"Justificación: {decision['decision_supervisor']['justificacion']}")
        print(f"Código de retorno: {decision['resultado_ejecucion']['returncode']}")
        print()
else:
    print("No se encontró el log de decisiones.")

Total de decisiones registradas: 11

--- Decisión 1 ---
Timestamp: 2026-10-28T07:03:07.526744
Acción: RUN_NEXT_STAGE
Justificación: La etapa 1 de inspección se completó exitosamente con código de retorno 0 y validación de contrato aprobada.
Código de retorno: 0

--- Decisión 2 ---
Timestamp: 2026-10-28T07:09:54.149722
Acción: REQUEST_CORRECTION
Justificación: La validación del contrato falló porque el JSON generado está incompleto y falta información crítica requerida por la especificación.
Código de retorno: 0

--- Decisión 3 ---
Timestamp: 2026-10-28T07:12:40.259494
Acción: REQUEST_CORRECTION
Justificación: El contrato de salida es inválido - faltan campos requeridos en el JSON de salida
Código de retorno: 0

--- Decisión 4 ---
Timestamp: 2026-10-28T07:15:08.049584
Acción: REQUEST_CORRECTION
Justificación: El script ejecutó correctamente pero el JSON de salida no cumple con el contrato esperado, faltan campos requeridos en la validación
Código de retorno: 0

--- Decisión 5 ---
Timest

#### Visualización de métricas de modelos

In [28]:
import pandas as pd

# Cargar comparación de modelos
comparacion_file = Path("data/comparacion_modelos.csv")
if comparacion_file.exists():
    df_comparacion = pd.read_csv(comparacion_file)
    print("Comparación de modelos:\n")
    print(df_comparacion.to_string(index=False))
else:
    print("No se encontró el archivo de comparación de modelos.")

Comparación de modelos:

               modelo                 clase                rol  train_RMSE  train_MAE  train_R2  train_MAPE  train_MedAE  test_RMSE  test_MAE   test_R2  test_MAPE  test_MedAE  sobreajuste  diferencia_r2
RandomForestRegressor RandomForestRegressor   modelo_principal    4.017179   3.015608  0.717766    3.681313     2.337276   4.968858  3.788897  0.607109   4.822881    3.001584        False       0.110656
                Ridge                 Ridge modelo_comparacion    7.465835   6.289208  0.025180    7.683375     5.329353   8.000746  6.646427 -0.018637   8.429361    5.743218        False       0.043817
       DummyRegressor        DummyRegressor         linea_base    7.672796   6.324786 -0.029615    7.855791     5.000000   8.441604  6.833333 -0.133988   8.788688    6.000000        False       0.104373


## **7. Discusión y mejoras futuras**

### 7.1 Limitaciones del hardware y de la ejecución local

El hardware disponible (4 GB VRAM + 16 GB RAM) impone restricciones:

- **Tamaño máximo de modelo**: ~9B parámetros cuantizados (`qwen3.5:9b`, ≈ 6.6 GB). Al superar los 4 GB de VRAM, parte del modelo se ejecuta en CPU/RAM.
- **Velocidad de inferencia**: 1-3 minutos por llamada LLM compleja, en parte por ese reparto entre GPU y CPU.
- **Ventana de contexto**: configurada a 16K tokens (`num_ctx=16384`) para evitar truncamiento del prompt, aunque el modelo admite más; una ventana mayor consume más memoria.
- **Paralelismo**: no se pueden ejecutar múltiples modelos simultáneamente.

### 7.2 Limitaciones del dataset

- **Tamaño**: 1198 filas es pequeño para modelos complejos.
- **Cardinalidad**: `city` (203), `venue` (335), `artist` (87) generan riesgo de sobreajuste con one-hot encoding.
- **Colinealidad**: posibles correlaciones entre variables no exploradas.
- **Calidad**: valores `???` en `pop`, nulos en `venue`, duplicados.
- **Documentación**: el repositorio original no define `pop` ni `score`, lo que limita la interpretación del negocio.

### 7.3 Variabilidad y limitaciones de los LLM

- **No determinismo**: aun con temperatura baja (el supervisor usa 0.1), los LLMs pueden producir salidas ligeramente distintas entre corridas; por eso no se garantiza la reproducibilidad exacta.
- **Alucinaciones**: el LLM puede citar cifras no presentes en los datos. Los scripts de etapa incluyen una verificación de cifras (`verificar_cifras`) que no se muestra en este notebook.
- **Errores de formato**: el LLM puede generar JSON inválido o truncado (se mitiga con reintentos y con `num_predict`).
- **Consumo de tokens por razonamiento**: los modelos con modo *thinking* (como Qwen3.5) pueden gastar parte del límite de salida en razonar antes de emitir el JSON.
- **Sesgos**: el LLM puede tener preferencias por ciertas técnicas (ej. one-hot) sin justificación estadística.

### 7.4 Confiabilidad de las decisiones y calidad de los resultados

El sistema implementa varias capas de validación:

- **Validación de contratos**: verifica archivos y campos JSON.
- **Validación de acciones**: el supervisor sólo puede elegir entre las acciones permitidas.
- **Validación de capacidades** y **verificación de cifras** (en los scripts de etapa): comprueban que las propuestas del LLM sean ejecutables y que los números citados existan en los datos.
- **Fallbacks conservadores**: si el LLM propone algo inválido, se usa una opción segura.

Sin embargo, la calidad final depende de las decisiones del LLM, que pueden ser subóptimas. Una mejora natural es que el resultado negativo de la validación del contrato impida avanzar, en lugar de ser sólo evidencia para el supervisor.

### 7.5 Mejoras en prompts y criterios de validación

- **Prompts más específicos**: añadir reglas cuantitativas (ej. "si filas/columnas < 5, evitar one-hot").
- **Validación estadística**: verificar que las decisiones del LLM sean estadísticamente razonables (ej. no proponer transformación log si hay valores negativos).
- **Métricas de calidad del prompt**: medir la tasa de reintentos y desviaciones por etapa.
- **Corrección dirigida**: que la acción `REQUEST_CORRECTION` transmita al agente qué debe corregirse, en lugar de repetir la misma ejecución.

### 7.6 Incorporación de modelos adicionales

- **Clasificadores**: el pipeline actual sólo soporta regresión. Añadir `LogisticRegression`, `RandomForestClassifier`, etc.
- **Modelos de series de tiempo**: si el dataset tiene componente temporal.
- **Modelos de clustering**: para tareas no supervisadas.

### 7.7 Soporte para datasets de mayor tamaño

- **Procesamiento por lotes**: dividir el dataset en chunks si excede la memoria.
- **Modelos más eficientes**: usar `HistGradientBoostingRegressor` en lugar de `GradientBoostingRegressor`.
- **Reducción de dimensionalidad**: PCA, UMAP para datasets con muchas características.

### 7.8 Implementación manual de ecuaciones y métricas

Para mayor control y transparencia, se podrían implementar manualmente:

- **Métricas personalizadas**: métricas específicas del dominio (ej. costo asimétrico de errores).
- **Validación cruzada estratificada**: para datasets desbalanceados (en clasificación).
- **Curvas de aprendizaje**: para diagnosticar sobreajuste/subajuste.

## **8. Conclusiones**

### 8.1 Principales logros

1. ✅ **Sistema multiagente funcional**: 4 etapas especializadas + 1 supervisor.
2. ✅ **Autonomía con validación**: los agentes proponen y deciden, pero el sistema valida contratos y acciones permitidas.
3. ✅ **Trazabilidad de decisiones**: las decisiones del supervisor y sus resultados se registran en JSON.
4. ✅ **Verificación de cifras**: los scripts de etapa comprueban que los números citados por el LLM existan en los datos.
5. ✅ **Pipeline modular**: cada etapa puede ejecutarse por separado. La reproducibilidad exacta no está garantizada por la variabilidad de los LLM (ver 7.3).

### 8.2 Cumplimiento del objetivo

El objetivo era construir un sistema multiagente que resolviera una problemática de ciencia de datos de forma autónoma. Se logró:

- El pipeline completó las 4 etapas sin intervención humana en las corridas realizadas por el autor (no se midió formalmente una tasa de éxito; una medición como "x de 10 corridas completas" fortalecería esta afirmación).
- Los agentes toman decisiones metodológicas (variable objetivo, encoding, modelos).
- El sistema valida y registra las decisiones.
- Los resultados son interpretables y trazables.

### 8.3 Aplicabilidad y trabajo futuro

- El pipeline puede adaptarse a otros datasets de regresión cambiando sólo `data/data.csv`.
- La arquitectura es modular: se pueden añadir etapas o modificar agentes.
- Si se desea puede ampliarse para soportar tareas de clustering.

## **9. Referencias bibliográficas**

### Arquitecturas y modelos

- https://www.ibm.com/es-es/think/topics/generative-ai

- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention is All You Need*. Advances in Neural Information Processing Systems, 30.

- Radford, A., Narasimhan, K., Salimans, T., & Sutskever, I. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.

- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. arXiv:1810.04805.

- Brown, T., et al. (2020). *Language Models are Few-Shot Learners*. Advances in Neural Information Processing Systems, 33.

- Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. Advances in Neural Information Processing Systems, 35. arXiv:2201.11903.

- Touvron, H., et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971.

- OpenAI. (2023). *GPT-4 Technical Report*. arXiv:2303.08774.

- Jiang, A. Q., et al. (2023). *Mistral 7B*. arXiv:2310.06825.

- Schaeffer, R., Miranda, B., & Koyejo, S. (2023). *Are Emergent Abilities of Large Language Models a Mirage?* Advances in Neural Information Processing Systems, 36. arXiv:2304.15004.

- Anthropic. (2024). *The Claude 3 Model Family: Opus, Sonnet, Haiku* (model card).

- Dubey, A., et al. (2024). *The Llama 3 Herd of Models*. arXiv:2407.21783.

- OpenAI. (2024). *OpenAI o1 System Card*. arXiv:2412.16720.

- Qwen Team. (2024). *Qwen2.5 Technical Report*. arXiv:2412.15115.

- DeepSeek-AI. (2025). *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning*. arXiv:2501.12948.

- Yang, A., et al. (2025). *Qwen3 Technical Report*. arXiv:2505.09388.

- Qwen Team. (2026). *Qwen3.5* [modelo de pesos abiertos]. Disponible en https://ollama.com/library/qwen3.5

### Ajuste eficiente de modelos

- Hu, E. J., et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. arXiv:2106.09685.

- Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs*. arXiv:2305.14314.

### Frameworks y herramientas

- Chase, H. (2022). *LangChain* [software]. https://www.langchain.com/

- Ollama. (2023). *Ollama: Get up and running with large language models locally* [software]. https://ollama.com/

- Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 12, 2825-2830.

### Agentes y sistemas multiagente

- Wooldridge, M., & Jennings, N. R. (1995). *Intelligent agents: Theory and practice*. The Knowledge Engineering Review, 10(2), 115-152.

- Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.

- Xi, Z., et al. (2023). *The Rise and Potential of Large Language Model Based Agents: A Survey*. arXiv:2309.07864.

- Anthropic. (2024). *Building effective agents*. https://www.anthropic.com/research/building-effective-agents

### Dataset

- ethanjaredlee. (2018). *ticketmaster-price-ml* [repositorio de GitHub]. https://github.com/ethanjaredlee/ticketmaster-price-ml (Licencia MIT).

### Ciencia de datos y machine learning

- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2nd ed.). Springer.

- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2nd ed.). O'Reilly Media.